# Bab 9: Pohon Keputusan dan Ensemble

Notebook pendamping buku *Pengenalan Pola dan Machine Learning: Pendekatan Probabilistik dengan Python*.

Jalankan sel dari atas ke bawah (**Runtime → Run all** di Colab). Sel pertama menyiapkan folder kode dan modul data sintetis; setiap sel berikutnya adalah satu listing dari buku, dengan nomor penanda `# (1)`, `# (2)`, … yang dijelaskan di bawah listing pada buku. Angka keluaran bisa sedikit berbeda dari buku bila versi pustakanya berbeda (lihat Lampiran A).

In [ ]:
# Menyiapkan folder kode/ dan modul pendukung bab ini
import os, pathlib
BERKAS = {
"bab09/adaboost_dari_nol.py": "import numpy as np\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import roc_auc_score\nfrom data_umkm import buat_data\n\nX, y = buat_data()\nXl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)\nt = 2 * yl - 1                                     # label -1/+1\n\ndef stump_terbaik(X, t, w):                      # pohon satu tingkat (stump)  #<1>\n    terbaik = (np.inf, None)\n    for j in range(X.shape[1]):\n        for ambang in np.unique(np.quantile(X[:, j], np.linspace(0.02, 0.98, 30))):\n            for arah in (1, -1):\n                pred = arah * np.where(X[:, j] > ambang, 1, -1)\n                galat = w[pred != t].sum()          # galat tertimbang\n                if galat < terbaik[0]:\n                    terbaik = (galat, (j, ambang, arah))\n    return terbaik\n\nw = np.full(len(t), 1 / len(t))                    # bobot awal seragam\nF_latih, F_uji = np.zeros(len(t)), np.zeros(len(yu))\nfor m in range(1, 201):\n    eps, (j, amb, arah) = stump_terbaik(Xl, t, w)\n    a = 0.5 * np.log((1 - eps) / eps)              # bobot pengklasifikasi  #<2>\n    h = arah * np.where(Xl[:, j] > amb, 1, -1)\n    w *= np.exp(-a * t * h); w /= w.sum()          # naikkan bobot yang salah  #<3>\n    F_latih += a * h\n    F_uji += a * arah * np.where(Xu[:, j] > amb, 1, -1)\n    if m in (1, 5, 20, 50, 200):\n        print(f\"M = {m:3d} | galat tertimbang {eps:.3f} | AUC latih \"\n              f\"{roc_auc_score(yl, F_latih):.3f} | uji {roc_auc_score(yu, F_uji):.3f}\")\n",
"bab09/boosting_kredit.py": "import numpy as np\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import roc_auc_score, log_loss\nfrom data_umkm import buat_data, FITUR\n\nX, y = buat_data()\nXl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)\nkategori = [FITUR.index(\"sektor\")]                 # sektor diperlakukan kategoris  #<1>\n\nprint(\" laju belajar | pohon dipakai | AUC uji | log-loss uji\")\nfor lr in (1.0, 0.3, 0.1, 0.03):                   # shrinkage (shrinkage)  #<2>\n    gb = HistGradientBoostingClassifier(learning_rate=lr, max_iter=2000, max_depth=3,\n                                        early_stopping=True, validation_fraction=0.15,\n                                        n_iter_no_change=30, categorical_features=kategori,\n                                        random_state=0).fit(Xl, yl)            #<3>\n    p = gb.predict_proba(Xu)[:, 1]\n    print(f\"{lr:13.2f} | {gb.n_iter_:13d} | {roc_auc_score(yu, p):7.3f} | {log_loss(yu, p):12.4f}\")\n",
"bab09/data_umkm.py": "\"\"\"Data pinjaman UMKM sintetis (ilustrasi). Sekitar 10% gagal bayar.\n\nHubungan fitur dengan risiko sengaja dibuat nonlinear dan berinteraksi,\nmisalnya rasio cicilan berbahaya terutama untuk usaha muda, dan pencatatan\ndigital (QRIS) menurunkan risiko di sektor perdagangan.\n\"\"\"\nimport numpy as np\n\nFITUR = [\"omzet_juta\", \"umur_usaha\", \"rasio_cicilan\", \"telat_12bln\",\n         \"pakai_qris\", \"karyawan\", \"sektor\", \"di_kota\"]\nSEKTOR = [\"kuliner\", \"dagang\", \"jasa\", \"kriya\", \"tani\"]\n\ndef buat_data(n=8000, seed=0):\n    rng = np.random.default_rng(seed)\n    omzet = np.exp(rng.normal(3.0, 0.8, n))                 # juta Rp per bulan\n    umur = rng.gamma(2.0, 2.5, n)                           # tahun\n    rasio = np.clip(rng.beta(2, 6, n), 0, 0.95)             # cicilan / laba\n    telat = rng.poisson(0.4, n)\n    qris = (rng.random(n) < 0.45).astype(int)\n    karyawan = rng.poisson(1 + omzet / 15)\n    sektor = rng.integers(0, 5, n)\n    kota = (rng.random(n) < 0.6).astype(int)\n    z = (-3.0\n         + 3.0 * (rasio > 0.4) * (1 + 1.5 * (umur < 2))        # ambang rasio cicilan\n         + 1.2 * np.minimum(telat, 3)                          # riwayat telat bayar\n         - 0.8 * (omzet > 15)                                  # omzet besar lebih aman\n         - 1.5 * qris * (sektor == 1)                          # QRIS membantu pedagang\n         - 0.8 * (umur > 5)\n         + 1.5 * (sektor == 4) * (1 - kota)                    # tani di desa: musiman\n         + rng.normal(0, 0.3, n))\n    y = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)\n    X = np.column_stack([omzet, umur, rasio, telat, qris, karyawan, sektor, kota])\n    return X, y\n",
"bab09/ensemble_kredit.py": "import numpy as np\nfrom sklearn.tree import DecisionTreeClassifier\nfrom sklearn.ensemble import BaggingClassifier, RandomForestClassifier\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import roc_auc_score\nfrom data_umkm import buat_data\n\nX, y = buat_data()\nXl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)\nauc = lambda m: roc_auc_score(yu, m.predict_proba(Xu)[:, 1])\n\npohon = DecisionTreeClassifier(min_samples_leaf=5, random_state=0).fit(Xl, yl)\nprint(f\"AUC uji satu pohon dalam: {auc(pohon):.3f}\")\nprint(\"  B | bagging (uji) | random forest (uji) | random forest (OOB)\")\nfor B in (25, 100, 500):                           # banyaknya pohon  #<1>\n    bag = BaggingClassifier(DecisionTreeClassifier(min_samples_leaf=5), n_estimators=B,\n                            random_state=0, n_jobs=-1).fit(Xl, yl)\n    rf = RandomForestClassifier(n_estimators=B, min_samples_leaf=5, max_features=\"sqrt\",\n                                oob_score=True, random_state=0, n_jobs=-1).fit(Xl, yl)  #<2>\n    oob = roc_auc_score(yl, rf.oob_decision_function_[:, 1])\n    print(f\"{B:3d} | {auc(bag):13.3f} | {auc(rf):19.3f} | {oob:19.3f}\")                                         #<3>\n",
"bab09/pohon_kredit.py": "import numpy as np\nfrom sklearn.tree import DecisionTreeClassifier, export_text\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import roc_auc_score\nfrom data_umkm import buat_data, FITUR\n\nX, y = buat_data()\nXl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)\nprint(f\"gagal bayar: {y.mean():.1%} dari {len(y)} pinjaman\")\n\nfor d in (1, 2, 4, 8, None):                       # kedalaman pohon  #<1>\n    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xl, yl)\n    auc = [roc_auc_score(t, m.predict_proba(A)[:, 1]) for A, t in ((Xl, yl), (Xu, yu))]\n    print(f\"kedalaman {str(d):4s}: {m.get_n_leaves():4d} daun | AUC latih {auc[0]:.3f} | uji {auc[1]:.3f}\")\n\npohon = DecisionTreeClassifier(max_depth=2, random_state=0).fit(Xl, yl)\nprint(export_text(pohon, feature_names=FITUR, show_weights=True, decimals=2))  #<2>\n",
"bab09/shap_dari_nol.py": "import numpy as np\nfrom itertools import combinations\nfrom math import factorial\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom sklearn.inspection import permutation_importance\nfrom sklearn.model_selection import train_test_split\nfrom data_umkm import buat_data, FITUR, SEKTOR\n\nX, y = buat_data()\nXl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)\ngb = HistGradientBoostingClassifier(learning_rate=0.1, max_iter=65, max_depth=3,\n                                    categorical_features=[6], random_state=0).fit(Xl, yl)\nf = lambda A: gb.decision_function(A)              # keluaran dalam log-odds\n\nimp = permutation_importance(gb, Xu, yu, scoring=\"roc_auc\", n_repeats=10, random_state=0)  #<1>\nurut = np.argsort(-imp.importances_mean)\nprint(\"kepentingan permutasi (turunnya AUC uji):\")\nprint(\"  \" + \", \".join(f\"{FITUR[j]} {imp.importances_mean[j]:.3f}\" for j in urut[:4]))\n\nx = Xu[np.argmax(f(Xu))]                           # pemohon paling berisiko di data uji\nlatar = Xl[np.random.default_rng(0).choice(len(Xl), 100, replace=False)]\nd = len(FITUR)\ndef v(S):                                          # nilai koalisi: fitur S dari x  #<2>\n    Z = latar.copy(); Z[:, list(S)] = x[list(S)]\n    return f(Z).mean()\nnilai = {S: v(S) for k in range(d + 1) for S in combinations(range(d), k)}\nphi = np.zeros(d)\nfor j in range(d):                                 # rumus Shapley eksak  #<3>\n    for S, vS in nilai.items():\n        if j not in S:\n            bobot = factorial(len(S)) * factorial(d - len(S) - 1) / factorial(d)\n            phi[j] += bobot * (nilai[tuple(sorted(S + (j,)))] - vS)\nprint(f\"pemohon: rasio {x[2]:.2f}, umur {x[1]:.1f} th, telat {int(x[3])}x, omzet {x[0]:.1f} jt, \"\n      f\"{SEKTOR[int(x[6])]}, {'kota' if x[7] else 'desa'}\")\nprint(f\"p(gagal) = {gb.predict_proba(x[None])[0, 1]:.2f}; dasar (rata-rata latar) {nilai[()]:+.2f} log-odds\")\nfor j in np.argsort(-np.abs(phi))[:5]:\n    print(f\"  {FITUR[j]:14s} {phi[j]:+.2f}\")\nprint(f\"dasar + jumlah SHAP = {nilai[()] + phi.sum():+.2f} = f(x) = {f(x[None])[0]:+.2f}\")  #<4>\n"
}
akar = pathlib.Path('pola-ml/kode').resolve()
for nama, isi in BERKAS.items():
    p = akar / nama; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(isi, encoding='utf-8')
os.chdir(akar / 'bab09')
print('folder kerja:', os.getcwd())

## Listing 9.1: Pohon keputusan dengan berbagai kedalaman

Berkas: `kode/bab09/pohon_kredit.py`

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from data_umkm import buat_data, FITUR

X, y = buat_data()
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
print(f"gagal bayar: {y.mean():.1%} dari {len(y)} pinjaman")

for d in (1, 2, 4, 8, None):                       # kedalaman pohon  # (1)
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xl, yl)
    auc = [roc_auc_score(t, m.predict_proba(A)[:, 1]) for A, t in ((Xl, yl), (Xu, yu))]
    print(f"kedalaman {str(d):4s}: {m.get_n_leaves():4d} daun | AUC latih {auc[0]:.3f} | uji {auc[1]:.3f}")

pohon = DecisionTreeClassifier(max_depth=2, random_state=0).fit(Xl, yl)
print(export_text(pohon, feature_names=FITUR, show_weights=True, decimals=2))  # (2)

## Listing 9.2: Satu pohon, bagging, dan random forest

Berkas: `kode/bab09/ensemble_kredit.py`

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from data_umkm import buat_data

X, y = buat_data()
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
auc = lambda m: roc_auc_score(yu, m.predict_proba(Xu)[:, 1])

pohon = DecisionTreeClassifier(min_samples_leaf=5, random_state=0).fit(Xl, yl)
print(f"AUC uji satu pohon dalam: {auc(pohon):.3f}")
print("  B | bagging (uji) | random forest (uji) | random forest (OOB)")
for B in (25, 100, 500):                           # banyaknya pohon  # (1)
    bag = BaggingClassifier(DecisionTreeClassifier(min_samples_leaf=5), n_estimators=B,
                            random_state=0, n_jobs=-1).fit(Xl, yl)
    rf = RandomForestClassifier(n_estimators=B, min_samples_leaf=5, max_features="sqrt",
                                oob_score=True, random_state=0, n_jobs=-1).fit(Xl, yl)  # (2)
    oob = roc_auc_score(yl, rf.oob_decision_function_[:, 1])
    print(f"{B:3d} | {auc(bag):13.3f} | {auc(rf):19.3f} | {oob:19.3f}")                                         # (3)

## Listing 9.3: AdaBoost dengan decision stump, dari nol

Berkas: `kode/bab09/adaboost_dari_nol.py`

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from data_umkm import buat_data

X, y = buat_data()
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
t = 2 * yl - 1                                     # label -1/+1

def stump_terbaik(X, t, w):                      # pohon satu tingkat (stump)  # (1)
    terbaik = (np.inf, None)
    for j in range(X.shape[1]):
        for ambang in np.unique(np.quantile(X[:, j], np.linspace(0.02, 0.98, 30))):
            for arah in (1, -1):
                pred = arah * np.where(X[:, j] > ambang, 1, -1)
                galat = w[pred != t].sum()          # galat tertimbang
                if galat < terbaik[0]:
                    terbaik = (galat, (j, ambang, arah))
    return terbaik

w = np.full(len(t), 1 / len(t))                    # bobot awal seragam
F_latih, F_uji = np.zeros(len(t)), np.zeros(len(yu))
for m in range(1, 201):
    eps, (j, amb, arah) = stump_terbaik(Xl, t, w)
    a = 0.5 * np.log((1 - eps) / eps)              # bobot pengklasifikasi  # (2)
    h = arah * np.where(Xl[:, j] > amb, 1, -1)
    w *= np.exp(-a * t * h); w /= w.sum()          # naikkan bobot yang salah  # (3)
    F_latih += a * h
    F_uji += a * arah * np.where(Xu[:, j] > amb, 1, -1)
    if m in (1, 5, 20, 50, 200):
        print(f"M = {m:3d} | galat tertimbang {eps:.3f} | AUC latih "
              f"{roc_auc_score(yl, F_latih):.3f} | uji {roc_auc_score(yu, F_uji):.3f}")

## Listing 9.4: Gradient boosting dengan early stopping

Berkas: `kode/bab09/boosting_kredit.py`

In [ ]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss
from data_umkm import buat_data, FITUR

X, y = buat_data()
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
kategori = [FITUR.index("sektor")]                 # sektor diperlakukan kategoris  # (1)

print(" laju belajar | pohon dipakai | AUC uji | log-loss uji")
for lr in (1.0, 0.3, 0.1, 0.03):                   # shrinkage (shrinkage)  # (2)
    gb = HistGradientBoostingClassifier(learning_rate=lr, max_iter=2000, max_depth=3,
                                        early_stopping=True, validation_fraction=0.15,
                                        n_iter_no_change=30, categorical_features=kategori,
                                        random_state=0).fit(Xl, yl)            # (3)
    p = gb.predict_proba(Xu)[:, 1]
    print(f"{lr:13.2f} | {gb.n_iter_:13d} | {roc_auc_score(yu, p):7.3f} | {log_loss(yu, p):12.4f}")

## Listing 9.5: Kepentingan permutasi dan nilai Shapley eksak, dari nol

Berkas: `kode/bab09/shap_dari_nol.py`

In [ ]:
import numpy as np
from itertools import combinations
from math import factorial
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from data_umkm import buat_data, FITUR, SEKTOR

X, y = buat_data()
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
gb = HistGradientBoostingClassifier(learning_rate=0.1, max_iter=65, max_depth=3,
                                    categorical_features=[6], random_state=0).fit(Xl, yl)
f = lambda A: gb.decision_function(A)              # keluaran dalam log-odds

imp = permutation_importance(gb, Xu, yu, scoring="roc_auc", n_repeats=10, random_state=0)  # (1)
urut = np.argsort(-imp.importances_mean)
print("kepentingan permutasi (turunnya AUC uji):")
print("  " + ", ".join(f"{FITUR[j]} {imp.importances_mean[j]:.3f}" for j in urut[:4]))

x = Xu[np.argmax(f(Xu))]                           # pemohon paling berisiko di data uji
latar = Xl[np.random.default_rng(0).choice(len(Xl), 100, replace=False)]
d = len(FITUR)
def v(S):                                          # nilai koalisi: fitur S dari x  # (2)
    Z = latar.copy(); Z[:, list(S)] = x[list(S)]
    return f(Z).mean()
nilai = {S: v(S) for k in range(d + 1) for S in combinations(range(d), k)}
phi = np.zeros(d)
for j in range(d):                                 # rumus Shapley eksak  # (3)
    for S, vS in nilai.items():
        if j not in S:
            bobot = factorial(len(S)) * factorial(d - len(S) - 1) / factorial(d)
            phi[j] += bobot * (nilai[tuple(sorted(S + (j,)))] - vS)
print(f"pemohon: rasio {x[2]:.2f}, umur {x[1]:.1f} th, telat {int(x[3])}x, omzet {x[0]:.1f} jt, "
      f"{SEKTOR[int(x[6])]}, {'kota' if x[7] else 'desa'}")
print(f"p(gagal) = {gb.predict_proba(x[None])[0, 1]:.2f}; dasar (rata-rata latar) {nilai[()]:+.2f} log-odds")
for j in np.argsort(-np.abs(phi))[:5]:
    print(f"  {FITUR[j]:14s} {phi[j]:+.2f}")
print(f"dasar + jumlah SHAP = {nilai[()] + phi.sum():+.2f} = f(x) = {f(x[None])[0]:+.2f}")  # (4)